# 02 — Dask graphs on the worker pool

`pyodide_pool` ships an async dask scheduler: `await pyodide_pool.compute(...)`
walks any dask graph and dispatches every ready task to the worker pool,
keeping up to `pool_size` tasks in flight. This notebook:

1. builds a `dask.delayed` graph and computes it on the pool,
2. runs a `dask.bag` pipeline the same way,
3. verifies both against dask's built-in `scheduler='synchronous'`,
4. demonstrates **package mirroring** with a numpy task — workers install
   what the kernel has installed before running your code.

> **Environment requirements** — same as `01-pool-basics.ipynb`: a
> cross-origin-isolated page (COOP/COEP headers, provided by
> `npm run serve:lite`), nested-worker support, and network access to the
> Pyodide CDN and PyPI.

## Install the wheel and dask

dask is pure Python, so piplite installs it from PyPI. The pool workers do
**not** need it preinstalled: package mirroring replays the kernel's
installs into each worker before the first task that ships there.

In [ ]:
import piplite

await piplite.install("/files/wheels/pyodide_pool-0.1.0-py3-none-any.whl")
await piplite.install("dask")

In [ ]:
import pyodide_pool

pool = await pyodide_pool.create_pool(pool_size=4)
print("pool ready with", pool.pool_size, "workers")

## The workload

The same trial-division prime counter as `01-pool-basics.ipynb`, over a
smaller range so the serial verification runs stay quick:
π(1,000,000) = 78,498.

In [ ]:
RANGE_START, RANGE_END = 2, 1_000_000
CHUNK_COUNT = 8
EXPECTED_TOTAL = 78_498  # π(1_000_000)


def count_primes(lo, hi):
    count = 0
    for n in range(lo, hi):
        if n < 2:
            continue
        if n == 2:
            count += 1
            continue
        if n % 2 == 0:
            continue
        d = 3
        is_prime = True
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 2
        if is_prime:
            count += 1
    return count


def make_chunks(start, end, count):
    width = -(-(end - start) // count)  # ceiling division
    return [(lo, min(lo + width, end)) for lo in range(start, end, width)]


chunks = make_chunks(RANGE_START, RANGE_END, CHUNK_COUNT)

## A `dask.delayed` graph

Eight `count_primes` leaves fan into one `sum` node. `await
pyodide_pool.compute(total)` is the async counterpart of `total.compute()`:
the scheduler dispatches all eight leaves to the pool at once, then runs
`sum` when they finish. The first pool run also mirrors dask into each
worker (the first-task-pays pattern), so it runs slower than steady state.

In [ ]:
import time

import dask

leaves = [dask.delayed(count_primes)(lo, hi) for lo, hi in chunks]
total = dask.delayed(sum)(leaves)

t0 = time.perf_counter()
pool_total = await pyodide_pool.compute(total)
pool_seconds = time.perf_counter() - t0
print(f"pool scheduler: {pool_total:,} primes in {pool_seconds:.2f} s")

## Verify against the synchronous scheduler

Same graph, dask's own single-threaded scheduler, right here in the kernel.

In [ ]:
t0 = time.perf_counter()
sync_total = total.compute(scheduler="synchronous")
sync_seconds = time.perf_counter() - t0

assert pool_total == sync_total == EXPECTED_TOTAL, (pool_total, sync_total)
print(f"synchronous: {sync_total:,} primes in {sync_seconds:.2f} s — results match")

## A `dask.bag` pipeline

Collections work too — the scheduler executes whatever graph the collection
compiles to. Here a bag of chunk tuples is mapped through `count_primes`
and reduced with `.sum()`.

In [ ]:
import dask.bag as db

bag_total = (
    db.from_sequence(chunks, npartitions=CHUNK_COUNT)
    .map(lambda chunk: count_primes(*chunk))
    .sum()
)

pool_bag = await pyodide_pool.compute(bag_total)
sync_bag = bag_total.compute(scheduler="synchronous")
assert pool_bag == sync_bag == EXPECTED_TOTAL, (pool_bag, sync_bag)
print(f"bag pipeline: {pool_bag:,} primes — matches the synchronous scheduler")

## Package mirroring: a numpy task

Importing numpy here loads it into the **kernel** from the Pyodide
distribution. `pyodide_pool` snapshots what the kernel has installed and
rides that snapshot on every task; each worker replays the installs before
unpickling — so `chunk_sum` below just works on the workers, with no
per-worker setup. The integer sum keeps the equality check exact.

In [ ]:
import numpy as np


def chunk_sum(lo, hi):
    import numpy as np

    return int(np.arange(lo, hi, dtype=np.int64).sum())


np_leaves = [dask.delayed(chunk_sum)(lo, hi) for lo, hi in chunks]
np_total = dask.delayed(sum)(np_leaves)

pool_np = await pyodide_pool.compute(np_total)
sync_np = np_total.compute(scheduler="synchronous")
expected_np = (RANGE_END * (RANGE_END - 1) - RANGE_START * (RANGE_START - 1)) // 2

assert pool_np == sync_np == expected_np, (pool_np, sync_np, expected_np)
print(f"numpy graph: {pool_np:,} — workers imported the mirrored numpy")

## Success marker

In [ ]:
assert pool_total == sync_total == EXPECTED_TOTAL
assert pool_bag == sync_bag == EXPECTED_TOTAL
assert pool_np == sync_np == expected_np
print("DASK_DEMO_OK")